# Baseline RAG - Turkish Legal QA

**Proje:** CENG493 Turkish Legal RAG  
**Adim:** 1/5 - Baseline RAG (Dense Only + Qwen2.5-7B)  
**GPU:** L4 (24GB) onerilen, T4 (16GB) de yeterli

| Bilesen | Model | Ayar |
|---------|-------|------|
| Embedding | BAAI/bge-m3 | Dense only (baseline) |
| Retrieval | FAISS Inner Product | Top-K |
| Reranker | Yok (baseline) | - |
| LLM | Qwen/Qwen2.5-7B-Instruct | 4-bit NF4 |

**Hazirlik:** Bu notebook'u calistirmadan once `493_project` klasorunu Google Drive'a yukleyin.  
Drive'daki yol: `MyDrive/493_project/`

**Korpus:** `data/processed/chunks.jsonl` dosyasi **`python scripts/prepare_corpus.py`** ile uretilmis olmali (Kaggle CSV ile hizali tam `context` chunklari). `chunk_corpus.py` ciktisi bu pipeline ile karistirilmamali.

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate bitsandbytes rouge-score nltk

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/493_project")

import sys
sys.path.insert(0, str(PROJECT_DIR))
from evaluation.metrics_utils import (
    RAG_SYSTEM_PROMPT,
    format_context_blocks_for_llm,
    aggregate_qa_row,
    retrieval_hit,
)

chunks = []
with open(PROJECT_DIR / "data/processed/chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

with open(PROJECT_DIR / "evaluation/gold_test_set.json", "r", encoding="utf-8") as f:
    gold_questions = json.load(f)

answerable_qs = [q for q in gold_questions if q.get("answerable_from_corpus", True)]

print(f"Chunks: {len(chunks)}")
print(f"Gold questions: {len(gold_questions)} total, {len(answerable_qs)} answerable")

## Embedding + FAISS Index

bge-m3 ile tum chunk'lari embed edip FAISS index'e ekliyoruz.  
Baseline icin sadece **dense** mod kullanilir (sparse ve colbert kapali).

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("BAAI/bge-m3", device="cuda")

chunk_texts = [c["text"] for c in chunks]
print(f"Encoding {len(chunk_texts)} chunks...")

dense_vecs = embed_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

dense_vecs = np.array(dense_vecs).astype("float32")
print(f"Embedding shape: {dense_vecs.shape}")

In [ ]:
import faiss

dim = dense_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(dense_vecs)

print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

save_dir = PROJECT_DIR / "models" / "baseline"
save_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(save_dir / "faiss.index"))
np.save(str(save_dir / "dense_vecs.npy"), dense_vecs)
print(f"Index saved -> {save_dir}")

## Retrieval

Sorguyu embed edip FAISS'ten en yakin K chunk'i getiriyoruz.

In [ ]:
def retrieve(query: str, top_k: int = 10) -> list:
    q_vec = embed_model.encode(
        [query], normalize_embeddings=True,
    )
    q_vec = np.array(q_vec).astype("float32")
    scores, indices = index.search(q_vec, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        c = chunks[idx]
        results.append({
            "rank": rank + 1,
            "score": float(score),
            "chunk_id": c["chunk_id"],
            "source": c["source"],
            "article_title": c.get("article_title", ""),
            "article_key": c.get("article_key", ""),
            "text": c["text"],
        })
    return results


print("Test: 'Egemenlik kime aittir?'\n")
for r in retrieve("Egemenlik kime aittir?", top_k=3):
    print(f"  Rank {r['rank']} ({r['score']:.4f}) [{r['source']} | {r['article_title']}]")
    print(f"  {r['text'][:120]}...\n")

## LLM + RAG Pipeline

Qwen2.5-7B-Instruct'i 4-bit quantization ile yukleyip RAG pipeline'i olusturuyoruz.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"LLM loaded  |  VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
def rag_answer(question: str, top_k: int = 5) -> dict:
    retrieved = retrieve(question, top_k)
    context = format_context_blocks_for_llm(retrieved, max_chunks=top_k)
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Baglam:\n{context}\n\nSoru: {question}"},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=4096
    ).to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
        )
    answer = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return {"question": question, "answer": answer.strip(), "retrieved": retrieved}


r = rag_answer("Egemenlik kime aittir?")
print(f"Soru : {r['question']}")
print(f"Cevap: {r['answer']}")

## Evaluation

150 answerable soru uzerinden baseline skorlarini hesapliyoruz.

**Retrieval:** Recall@K, MRR, nDCG@10 (gold_context overlap >= 0.4)  
**QA:** F1, EM, BLEU (smoothed), ROUGE-L, faithfulness (token recall vs baglam), atif (retrieved precision + gold reference_chunk_ids recall).  
**Cevap formati:** `evaluation/metrics_utils.RAG_SYSTEM_PROMPT` — Cevap + Kaynaklar + [chunk_id=...] (CENG493).

In [ ]:
import nltk

for _p in ("punkt", "punkt_tab"):
    try:
        nltk.download(_p, quiet=True)
    except Exception:
        pass

print("Eval: retrieval_hit + aggregate_qa_row -> evaluation/metrics_utils.py")

In [ ]:
from tqdm import tqdm

TOP_K_RETRIEVAL = 10
TOP_K_CONTEXT = 5

results = []
for q in tqdm(answerable_qs, desc="Evaluating"):
    ret = retrieve(q["soru"], top_k=TOP_K_RETRIEVAL)
    rh = retrieval_hit(ret, q)

    ra = rag_answer(q["soru"], top_k=TOP_K_CONTEXT)
    qa = aggregate_qa_row(ra["answer"], q["cevap"], ra["retrieved"], q)
    pred_body = qa.pop("pred_body")
    results.append({
        "id": q["id"],
        "soru": q["soru"],
        "gold_cevap": q["cevap"],
        "pred_cevap": ra["answer"],
        "pred_body": pred_body,
        "kaynak": q["kaynak"],
        "difficulty": q["difficulty"],
        "type": q["type"],
        **qa,
        **rh,
    })

print(f"\nEvaluation complete: {len(results)} questions")

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

print("=" * 60)
print("  BASELINE RAG SONUCLARI")
print("=" * 60)

print("\n--- Retrieval Metrics ---")
for k in [1, 3, 5, 10]:
    v = df[f"hit@{k}"].mean()
    print(f"  Recall@{k:<2d} : {v:.4f}  ({v*100:.1f}%)")
print(f"  MRR       : {df['mrr'].mean():.4f}")
print(f"  nDCG@10   : {df['ndcg@10'].mean():.4f}")

print("\n--- QA Metrics ---")
print(f"  F1        : {df['f1'].mean():.4f}")
print(f"  EM        : {df['em'].mean():.4f}")
print(f"  BLEU      : {df['bleu'].mean():.4f}")
print(f"  ROUGE-1   : {df['rouge1'].mean():.4f}")
print(f"  ROUGE-2   : {df['rouge2'].mean():.4f}")
print(f"  ROUGE-L   : {df['rougeL'].mean():.4f}")
print(f"  Faithful. : {df['faithfulness_token_recall'].mean():.4f}")
print(f"  CitePrec  : {df['citation_precision_retrieved'].mean():.4f}")
_cr = df.loc[df['citation_recall_gold'] >= 0, 'citation_recall_gold']
print(f"  CiteRecG  : {_cr.mean():.4f}  (yalnizca reference_chunk_ids dolu sorular, n={len(_cr)})")

print("\n--- Difficulty ---")
for d in ["easy", "medium", "hard", "very hard"]:
    s = df[df["difficulty"] == d]
    if len(s):
        print(f"  {d:10s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  R-L={s['rougeL'].mean():.3f}  (n={len(s)})")

print("\n--- Question Type ---")
for t in sorted(df["type"].unique()):
    s = df[df["type"] == t]
    print(f"  {t:15s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  R-L={s['rougeL'].mean():.3f}  (n={len(s)})")

In [ ]:
_cr = df.loc[df["citation_recall_gold"] >= 0, "citation_recall_gold"]

output = {
    "experiment": "baseline_dense_only",
    "embedding": "BAAI/bge-m3",
    "llm": "Qwen/Qwen2.5-7B-Instruct-4bit",
    "retrieval": "dense_only",
    "reranker": None,
    "top_k_retrieval": TOP_K_RETRIEVAL,
    "top_k_context": TOP_K_CONTEXT,
    "num_questions": len(results),
    "metrics": {
        "recall@1": float(df["hit@1"].mean()),
        "recall@3": float(df["hit@3"].mean()),
        "recall@5": float(df["hit@5"].mean()),
        "recall@10": float(df["hit@10"].mean()),
        "mrr": float(df["mrr"].mean()),
        "ndcg@10": float(df["ndcg@10"].mean()),
        "f1": float(df["f1"].mean()),
        "em": float(df["em"].mean()),
        "bleu": float(df["bleu"].mean()),
        "rouge1": float(df["rouge1"].mean()),
        "rouge2": float(df["rouge2"].mean()),
        "rougeL": float(df["rougeL"].mean()),
        "faithfulness_token_recall": float(df["faithfulness_token_recall"].mean()),
        "citation_precision_retrieved": float(df["citation_precision_retrieved"].mean()),
        "citation_recall_gold_mean": float(_cr.mean()) if len(_cr) else None,
    },
    "per_question": results,
}

save_path = PROJECT_DIR / "evaluation" / "baseline_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Results saved -> {save_path}")
print("\nBaseline tamamlandi! Bu skorlar sonraki deneylerin referans noktasi olacak.")